
# Exploratory Data Analysis & Preprocessing 


## Imports

In [41]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder
from sklearn.impute import SimpleImputer
from scipy import sparse

import warnings
warnings.filterwarnings("ignore")


## Load

In [42]:
df = pd.read_csv("../data/raw/rents.csv")
print(df.shape)
df.head(3)


(11657, 8)


,address,district,area,bedrooms,garage,type,rent,total
0,Rua Herval,Belenzinho,21,1,0,Studio e kitnet,2400,2939
1,Avenida São Miguel,Vila Marieta,15,1,1,Studio e kitnet,1030,1345
2,Rua Oscar Freire,Pinheiros,18,1,0,Apartamento,4000,4661


## Target and feature columns

In [43]:
TARGET = 'total'
assert TARGET in df.columns, f"Target column '{TARGET}' not found in CSV columns: {df.columns.tolist()}"

y = df[TARGET].copy()
X = df.drop(columns=[TARGET]).copy()

# Identify basic dtypes
num_selector = make_column_selector(dtype_include=np.number)
cat_selector = make_column_selector(dtype_include=object)

num_cols = num_selector(X)
cat_cols = cat_selector(X)

print('Numeric cols:', num_cols)
print('Categorical cols:', cat_cols)


Numeric cols: ['area', 'bedrooms', 'garage', 'rent']
Categorical cols: ['address', 'district', 'type']


## Stratify by price tiers to preserve distribution of the target in train/test

In [44]:
q = pd.qcut(y, q=5, duplicates='drop')
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=q
)
print(X_train.shape, X_test.shape)


(9325, 7) (2332, 7)


# Helper functions

#

In [45]:
def clip_upper(arr, upper):
    if sparse.issparse(arr):
        arr = arr.toarray()
    return np.clip(arr, None, upper)

def to_dense_if_sparse(Xm):
    return Xm.toarray() if sparse.issparse(Xm) else Xm

def normalize_text_series(s: pd.Series) -> pd.Series:
    s = s.copy()
    mask = s.notna()
    s.loc[mask] = s.loc[mask].astype(str).str.strip().str.lower()
    return s

def normalize_text_df(df_in: pd.DataFrame) -> pd.DataFrame:
    df_out = df_in.copy()
    for c in df_out.columns:
        df_out[c] = normalize_text_series(df_out[c])
    return df_out


## Area p99 clipping

In [46]:
AREA_COL = 'area'
if AREA_COL in X_train.columns:
    p99_area = pd.to_numeric(X_train[AREA_COL], errors='coerce').quantile(0.99)
    print('p99_area =', float(p99_area))
else:
    p99_area = None
    print(f"[info] Column '{AREA_COL}' not in data; skipping area clipping.")


p99_area = 387.28000000000065


# Build pipelines

In [47]:
num_pipe = Pipeline(steps=[
    ('imp', SimpleImputer(strategy='median')),
])

cat_pipe = Pipeline(steps=[
    ('normalize', FunctionTransformer(normalize_text_df, feature_names_out='one-to-one', validate=False)),
    ('imp', SimpleImputer(strategy='constant', fill_value='missing')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))  
])

if p99_area is not None:
    area_pipe = Pipeline(steps=[
        ('imp', SimpleImputer(strategy='median')),
        ('clip', FunctionTransformer(lambda x: clip_upper(x, p99_area), validate=False)),
    ])
    transformers = [
        ('num', num_pipe, [c for c in num_cols if c != AREA_COL]),
        ('cat', cat_pipe, cat_cols),
        ('area', area_pipe, [AREA_COL]),
    ]
else:
    transformers = [
        ('num', num_pipe, num_cols),
        ('cat', cat_pipe, cat_cols),
    ]

ct = ColumnTransformer(transformers=transformers, remainder='drop', sparse_threshold=1.0)


## Fit/transform

In [48]:
Xtr = ct.fit_transform(X_train)
Xte = ct.transform(X_test)

Xd = to_dense_if_sparse(Xtr)
has_nan_train = np.isnan(Xd).any()

Xe = to_dense_if_sparse(Xte)
has_nan_test = np.isnan(Xe).any()

print('Has NaN? train:', has_nan_train, ' test:', has_nan_test)


Has NaN? train: False  test: False


## Save splits

In [49]:
train_df = X_train.copy()
train_df[TARGET] = y_train.values

test_df = X_test.copy()
test_df[TARGET] = y_test.values

train_path = '../data/work/train.csv'
test_path  = '../data/work/test.csv'

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

train_path, test_path


('../data/work/train.csv', '../data/work/test.csv')